In [ ]:
import numpy as np
import matplotlib.pyplot as plt
dof = 10
x = np.random.randn(dof)/2 # np.arange(10)
def make_polynomial(vector, _dof=None):
    if not _dof: _dof = dof
    b = vector.shape[0]
    return np.cumprod(np.concatenate([np.ones((b, 1)), vector.reshape(-1, 1).repeat(_dof-1, axis=1)], axis=1), axis=1)
Z = make_polynomial(x, 10)
d = Z @ np.random.randn(10)
plt.title("Random Points")
plt.scatter(x, d)

In [ ]:
model = np.random.randn(dof)
print("+".join([f"{pi}x^{degree}" for degree, pi in enumerate(model)]))
def loss(model): return 0.5 * (Z @ model  - d) ** 2
print(f"{loss(model).mean()=}")

def make_polynomial_plot(model, x, d, dof=10, total_plots=5):
    fig, ax = plt.subplots(1, total_plots+1, figsize=(15, 5))
    space = np.linspace(-1.96, 1.96, 100) # 2sigma?
    for degree in range(dof)[:total_plots]:
        pi = model[degree]
        graph = space ** degree * pi
        ax[degree].plot(space, graph)
        ax[degree].scatter(x, pi * x ** degree, marker="*", color="r")
        ax[degree].set_title(f"Degree {degree}")
        ax[degree].set_ylim(-0.5, 0.5)
    ax[total_plots].set_title("Random Points")
    ax[total_plots].plot(space, make_polynomial(space, dof) @ model)
    ax[total_plots].scatter(x, d, marker="*", color="orange")
    ax[total_plots].set_ylim(-2, 2)
    fig.tight_layout()
    return

_ = make_polynomial_plot(model, x, d)

In [ ]:
# minimize 1/2(model(Z)-d)^2
# dL/dw => below ((Z @ model - d) * Z).mean(axis=0)
model = np.random.randn(10)
alpha = 0.1
make_polynomial_plot(model, x, d, total_plots=3)
for i in range(200):
    if i % 10 == 0:
        print(loss(model).mean())
    model -= alpha * ((Z @ model - d).reshape(-1, 1) * Z).mean(axis=0)
make_polynomial_plot(model, x, d, total_plots=3)

**simplify the situation here, for getting insights.**

find the 1 degree of polynomial line crossing (0, 1) (1, 0).

$f(w) = 0.25 \times ((p_0-1)^2+(p_0+p_1-0)^2) = 0.5 \times w^T A w - b^Tw + C$

where, w=coeff; A=[[4, 2], [2, 2]]; b=[2, 0]

In [ ]:
x = np.array([0, 1])
y = np.array([1, 0])
Z = make_polynomial(x, _dof=2)
w = np.random.randn(2) # coefficients of polynomial
A = np.array([[2, 1], [1, 1]]) / 2
b = np.array([1, 0]) / 2
Ainv = np.linalg.inv(A)
eigval, eigvec = np.linalg.eig(A) # Q \lambda Q^T -> ECA

As we get $\nabla f(w)=Aw-b$, $w^*=A^{-1}b$

In [ ]:
w_optimal = Ainv @ b
w_optimal

In [ ]:
eigvec, eigval, eigvec.T @ eigvec

A @ eigvec, eigvec @ np.diag(eigval)

In [ ]:
alpha = 0.1
w0 = w
x0 = eigvec.T @ (w0-w_optimal) # deformation along the eigvector
w1 = w0 - alpha * (A @ w0 - b)
x1 = eigvec.T @ (w1-w_optimal)
w2 = w1 - alpha * (A @ w1 - b)
x2 = eigvec.T @ (w2-w_optimal)

np.allclose(x0 * (1-alpha*eigval), x1), np.allclose(x0 * (1-alpha*eigval)**2, x2)

In [ ]:
np.allclose(w1, w0 - alpha * ((Z @ w0 - y).reshape(-1, 1) * Z).mean(axis=0))

In [ ]:
plt.bar(range(2), (1-alpha*eigval))
plt.title("Exponential Shrink Rate")

In [ ]:
alpha_optimal = 2 / (eigval[0] + eigval[1])
plt.bar(range(2), (1-alpha_optimal*eigval))
plt.title("Exponential Shrink Rate, alpha is optimal")

In [ ]:
ww = w0.copy()
for i in range(7):
    ww -= alpha_optimal * (A @ ww - b)
make_polynomial_plot(ww, x, y, dof=2, total_plots=2)

**Extend to Nth polynomial data**

Following FFT's idea we get (n^2->nlogn), but we can do more with O(1). How is it possible? This is math not algorithm.

In 2 dimensional A, we can easily get a coeff landscape information. How to extend to N data points specified? -> 이항정리

- Calculate A, b ~ N^3
- Calculate Eigvector ~ QR Decomposition

We get kth gradient descent product with closed-form expression. Despite of some costy preparations... ORR Calculate the w_optimal for happiness!

In [ ]:
dof = 10
x = np.random.randn(dof)/2
Z = make_polynomial(x, dof)
d = Z @ np.random.randn(10)

A = (Z.T @ Z) / dof
b = Z.T @ d / dof
eigval,eigvec = np.linalg.eig(A)
w_optimal = np.linalg.inv(A) @ b
np.allclose(eigvec @ np.diag(1/eigval) @ eigvec.T, np.linalg.inv(A))

In [ ]:
(1-alpha_optimal*eigval)

In [ ]:
alpha_optimal = 2 / (eigval[0] + eigval[-1])
plt.plot(range(10), 1-alpha_optimal*eigval, color="b")
plt.plot(range(10), 1-0.1*eigval, color="r")
plt.axhline(-1, 0, 1, color='gray', linestyle='--', linewidth=3)
plt.title("Exponential Shrink Rate, blue: alpha is optimal")

In [ ]:
w0 = np.random.randn(dof)
k = 5000
x0 = eigvec.T @ (w0-w_optimal) # deformation along the eigvector
xk = x0 * (1-alpha_optimal*eigval) ** k
wk = eigvec @ xk + w_optimal

In [ ]:
make_polynomial_plot(wk, x, d, dof)

In [ ]:
make_polynomial_plot(w_optimal, x, d, dof)

In [ ]:
import math
## FFT - https://numpy.org/doc/2.2/reference/routines.fft.html
### odd/even function decomposition
# f(w) +w g(w) or f(w) -w g(w)
# frequency space <-> signal space
def get_w(theta):
    return np.cos(theta) + np.sin(theta) * 1j

def ft(coeff):
    rank = len(coeff)
    theta = 2 * math.pi / rank
    H = make_polynomial(get_w(np.arange(rank) * theta), rank)
    return H @ coeff

def fft(coeff, start=True, inverse=False):
    rank = len(coeff)
    if start:
        pad = 2 ** math.ceil(math.log2(rank))
        coeff = np.pad(coeff, (0, pad-rank), mode="constant", constant_values=0)
    padded_coeff_rank = len(coeff)
    w = get_w(2 * math.pi / len(coeff))
    w = 1 / w if inverse else w # Hf
    if len(coeff) == 1:
        return coeff
    even = fft(coeff[::2], start=False)
    odd = fft(coeff[1::2], start=False)

    frequencies = np.zeros(padded_coeff_rank, dtype=np.complex64)
    half = padded_coeff_rank // 2
    for i in range(half):
        frequencies[i] = even[i] + w ** i * odd[i]
        frequencies[i + half] = even[i] - w ** i * odd[i]
    if start:
        divider = 1
        if inverse:
            divider = padded_coeff_rank # Hermitian operation
        return frequencies[:rank] / divider
    return frequencies

coeff = np.array([1, 2, 3, 4])
print(np.allclose(ft(coeff), fft(coeff)))

fig, ax1 = plt.subplots()
ax1.plot(range(len(coeff)), fft(coeff).real, color="blue", label="real")
ax1.set_ylabel('(real)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax2 = ax1.twinx()
ax2.plot(range(len(coeff)), (-fft(coeff) * 1j).real, color="red", label="complex")
ax2.set_ylabel('(complex)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

np.fft.ifft(np.fft.fft(coeff)), fft(fft(coeff), inverse=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

# Differential Equation => Iterative Gradient Descent
#       gradient = Ax (if b=0)
# x' = -2x -1y
# y' = -1x -1y
# e^{Rt} => position after time t passed.
R = np.array([[-2, -1], [-1, -1]], dtype=np.float64)
eigval, eigvec = np.linalg.eig(R)
# R /= np.linalg.norm(R, axis=0)

def exp_of_matrix(A, t):
    result = np.zeros_like(A, dtype=np.float64)
    for i in range(10): # taylor expansion
        result += t**i * np.linalg.matrix_power(A, i) / np.cumprod(np.arange(1, max(1, i)+1))[-1]
    return result

def get_trajectory(pos_initial, color):
    poses = []
    poses.append(pos_initial)
    for i in range(30):
        t = (i+1)/100
        # Rt = exp_of_matrix(R, t)
        Rt = expm(R*t)
        pos_t = Rt @ poses[-1]
        poses.append(pos_t)
    poses = np.stack(poses, 0)
    plt.scatter(poses[:, 0], poses[:, 1], color=color)

x = np.linspace(-1, 1, 100)
scope = eigvec[1, :] / eigvec[0, :]
get_trajectory(np.array([0, 1]), color="b")
get_trajectory(np.array([1, 0]), color="green")
get_trajectory(np.array([-1, -1]), color="c")
get_trajectory(np.array([-.5, .3]), color="orange")
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.title("Linear System Overview")

In [ ]:
x = np.linspace(-1, 1, 100)
scope = eigvec[1, :] / eigvec[0, :]
plt.plot(x, x*scope[0], color="red", label=eigval[0])
plt.plot(x, x*scope[1], color="blue", label=eigval[1])
get_trajectory(eigvec[:, 0], color="gray")
get_trajectory(eigvec[:, 1], color="gray")
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.title("Linear System Overview")
plt.legend()

$f(w) = 0.5 \times w^T A w - b^Tw$

A is Hessian of f(w). Laplacian is tr(H)=tr(A).

If A is not positive definite(xTAx>0, for all A), the landscape has saddle points.

- Hessian is symmetry (partial derivative)
- PDF is good-defined guy.
- PDF ensures all \lambda > 0 (definition)

https://gregorygundersen.com/blog/2022/02/27/positive-definite/

+) 

In [ ]:
A = np.array([[2, 1], [1, 1]]) # (2, 2)

def draw_hessian(A):
    xi = yi = np.linspace(-2, 2, 100)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = np.zeros_like(Xi)
    for i in range(100):
        for j in range(100):
            xy = np.array([Xi[i][j], Yi[i][j]])
            z = 0.5 * xy.T @ A @ xy
            Zi[i][j] = z

    contour = plt.contourf(Xi, Yi, Zi, levels=15, cmap='RdBu_r', linestyles="--")
    plt.arrow(0, 0, *(eigvec[:, 0]*eigval[0]), color="white", linewidth=2)
    plt.arrow(0, 0, *(eigvec[:, 1]*eigval[1]), color="white", linewidth=2) 
    plt.xlim(-2, 2)
    plt.ylim(-2, 2)

draw_hessian(A)

In [ ]:
A = np.array([[1, -2], [-2, 1]])
draw_hessian(A)

Graph Laplacian - https://math.stackexchange.com/questions/4089508/intuition-of-the-connection-between-the-graph-laplacian-and-the-laplace-operator

*Quick View*

- The 2nd derivatives helps us figuring out the key of "law of nature" (Laplace equation: Lxx+Lyy=0)
    - [minimal surfaces](https://www.youtube.com/watch?v=8SABptOYUVk&t=15s): Langrangian(변분)
    - $m \frac{d^2x}{dt^2}=-k(-x_{i-1}+2x_i-x_{1+1})$ -> Laplacian
    - harmonic property: averaging of neighbor's
- Graph in a linear algebric view!
     - discrete laplace operator = approx of continuous with finite difference method

ps) Any conn with Laplace Transform?

**Spectral Partitioning**

C = C(G): Incidence matrix

Graph Laplacian, L(G) = C^TC (loss the direction)

L(G) = D(egree) - W(adjacency)

- L(G) is symmetry, positive-semidefinite(wwT!->|xw|^2 ----- eigenvalue can be 0 $\lambda_i \geq 0$)
- L(G) has real-valued non-negative eigenvalues(PDF) and real-valued orthogonal eigenvectors
    - symmetry PD -> real-valued eigenvectors (consequence of spectral theorem) - Hermitian matrices and more details...
- G has K connected components if and only if $\lambda_0=...=\lambda_{k-1}=0$
    - Laplacian makes ${(f(i)-f(j))}^2$ where $i, j \in Edges$, this 
- Cutting edge = 1/4 xTL(G)x (1/2 for start-end symmetry and 1/2 for in and out double counting) -> How to make a clustering! (partition graph with this loss term.)
    - Fiedler eigenvector says the argmin xTLx. (ordered by significant level, reverse with PCA case high eig=important)

*Related*

- GOD: https://www.youtube.com/watch?v=Vng9lkibGEE
- Kirchoff's theorem for calculating number of spanning tree
- Fiedler vector for sparset cut
- Spectral decomposition
- Graph-based signal processing(graph-FT?)

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

nodes = list(range(6))
edges = [(0, 1), (0, 4), (1, 2), (1, 4), (2, 3), (3, 4), (3, 5)]
G = nx.Graph()
G.add_edges_from(edges)
G.graph["name"] = "normal"

ugly_edges = list(zip(nodes, nodes[1:]))
ugly_graph = nx.Graph()
ugly_graph.add_edges_from(ugly_edges)
ugly_graph.graph["name"] = "ugly"

In [ ]:
G.adj, G.degree

In [ ]:
import seaborn as sns

def awesome_graph(G, w:np.array, pos=None, return_pos=False):
    f, ax = plt.subplots(figsize=(8, 6))
    division = 20
    palette = sns.diverging_palette(250, 30, l=65, center="dark", as_cmap=True)(np.linspace(0, 1, division))
    node_color = []
    node_w = (np.clip(w*division/2, -division/2+1, division/2-1)+division/2).astype(np.int8)
    for n in G.nodes:
        node_color.append(tuple(palette[node_w[n]]))

    options = {
        "font_size": 36,
        "node_size": 3000,
        "node_color": node_color,
        "edgecolors": "black",
        "font_color": "white",
        "linewidths": 5,
        "width": 5,
    }
    if not pos:
        pos = nx.spring_layout(G)
    _ = nx.draw(G, with_labels=True, **options, ax=ax, pos=pos)
    if return_pos:
        return ax, pos
    return ax

awesome_graph(G, np.random.randn(len(G)))

In [ ]:
pos = nx.spring_layout(G, dim=2, seed=779)
node_xy = np.array([pos[v] for v in sorted(G)])
edge_xy = np.array([(pos[u], pos[v]) for u, v in G.edges()])


for e in edge_xy:
    plt.plot(*e.T, color="tab:gray")
plt.scatter(*node_xy.T, s=300, color="black")
for x, y, label in zip(*node_xy.T, sorted(G)):
    plt.text(x-0.02, y-0.03, label, {"color": "white", "size": 15})

plt.axis("off")

In [ ]:
adj = np.array([[0, 1, 0, 0, 1, 0],
                [1, 0, 1, 0, 1, 0],
                [0, 1, 0, 1, 0, 0],
                [0, 0, 1, 0, 1, 1],
                [1, 1, 0, 1, 0, 0],
                [0, 0, 0, 1, 0, 0]
            ])
degree = np.diag(adj.sum(1))
graph_laplacian = degree - adj

edges = G.edges
def get_incidence_matrix(edges):
    total_edges = len(edges)
    incidence_matrix = np.zeros((total_edges, len(G)))
    starts = [edge[0] for edge in edges]
    ends = [edge[1] for edge in edges]
    incidence_matrix[range(total_edges), starts] = 1
    incidence_matrix[range(total_edges), ends] = -1
    return incidence_matrix

def get_graph_laplacian_from_incidence_matrix(incidence_matrix):
    return incidence_matrix.T @ incidence_matrix

incidence_matrix = get_incidence_matrix(edges)
ugly_graph_laplacian = get_graph_laplacian_from_incidence_matrix(get_incidence_matrix(ugly_edges))

np.all(graph_laplacian == get_graph_laplacian_from_incidence_matrix(incidence_matrix)) # L(G) = CTC

In [ ]:
x = np.array([1, 1, -1, -1, 1, -1])
print(0.25 * x.T @ graph_laplacian @ x)
awesome_graph(G, x)

In [ ]:
w = np.array([1, 0, 0, 0, 0, 0])
awesome_graph(G, w)

In [ ]:
awesome_graph(G, np.linalg.matrix_power(graph_laplacian, 2) @ w)

In [ ]:
import torch
def get_sub_optimal_alpha(A):
    eigval, eigvec = np.linalg.eig(A)
    sub_optimal_alpha = 2 / (eigval[-1] + eigval[0])
    return sub_optimal_alpha

def train_graph(G, graph_laplacian):
    torch_graph_L = torch.from_numpy(graph_laplacian).to(torch.float)
    data = torch.tensor([1, 0, 0, 0, 0, 0], dtype=torch.float)
    data.requires_grad_()
    parameters = [data]
    lr = get_sub_optimal_alpha(graph_laplacian)
    pos = None
    for i in range(10):
        loss = 1/2 * data.T @ torch_graph_L @ data + (data[0] - 1) ** 2 # physical duality situation = sound source
        loss.backward()
        for p in parameters:
            p.data -= p.grad * lr
        print(loss.item(), data)
        _, pos = awesome_graph(G, data.detach().numpy(), pos=pos, return_pos=True)
        plt.savefig(f"./natural_gradient/{G.graph['name']}_{i}.jpeg")
train_graph(G, graph_laplacian)

In [ ]:
train_graph(ugly_graph, ugly_graph_laplacian)

In [ ]:
eigval, eigvec = np.linalg.eig(graph_laplacian)
w = eigvec[:, 1] # graph partitioning optimal w*
print(w)
awesome_graph(G, w)

In [ ]:
import torch

torch_graph_L = torch.from_numpy(graph_laplacian).to(torch.float)
data = torch.randn(6, dtype=torch.float)
data /= torch.norm(data)
data.requires_grad_()
parameters = [data]
lr = 0.1
for i in range(20):
    loss = 1/4 * data.T @ torch_graph_L @ data + 0.1 * (torch.norm(data)-1)**2
    loss.backward()
    for p in parameters:
        p.data -= p.grad * lr
    # print(loss.item())

print(f"Opt: {w} Train: {data}")
print(f"Optimal Loss: {1/4*w.T@graph_laplacian@w=}")
print(f"Train Loss: {loss.item()}")
print(f"Diff norm: {((torch.tensor(w)-data)**2).sum()}")

awesome_graph(G, data.detach().numpy())

In [ ]:
# what is spectral layout?
#   lambda2 and lambda3 being used to represent x-y coordinate!
#   -> -/+ partitioning. building 4 groups with confidence value. (near 0 value is thought as bridging between groups.)
awesome_graph(G, w, nx.spectral_layout(G))

Page Rank(=markov process) follows the similar logic path above. 

- page rank matrix is just transition matrix (prob of page j to page i)
- it is non-symmetry

in real world case G = dM + (1-d)E/N -> random walk!

> The process can be modeled as a random walk on the web graph.

"t"th user expected desitination prob distribution = $M^t x$

all the $0 \leq \lambda_i \leq 1$ as it is transition matrix (remind the axiom of probability)

In [ ]:
page_rank = np.array([[0, 0, 1, 1/2], [1/2, 0, 0, 0], [1/2, 1, 0, 1/2], [0, 0, 0, 0]])
(m:=np.linalg.matrix_power(page_rank, 100)), m@np.ones(4)/4

In [ ]:
(m:=np.linalg.matrix_power(page_rank, 1000)), m@np.ones(4)/4

If we think a little bit, you come up with why this phenomenon occurs. **Fixed Point!**

$\frac{dx_t}{dt}=(A-I)x_t; \frac{dx_t}{dt}=0 \to x_t=w*$ 

(A-I)x is equals to +in bound/-out bound.

w* would be (no need to solve D.E.)

- eigen vec whose val is equal to 1
- OR permutate relationship

The speed of convergence is depend on the ratio of largest and semi-largest eigenvalue!

*Related*

- Google's secret: https://rstudio-pubs-static.s3.amazonaws.com/239261_8a607707294341c4b7e26acf728c28bd.html
- Kolmogorov forward equation (expands to continuous)
- Krylov Subspace: well-defined property of finite space.(power method)

In [ ]:
w = np.linalg.eig(page_rank)[1][0]

Explore more on weigthed & undirected (ugly) ones.

- Symmetry weighted case = just $w_{ij}(f(i)-f(j))^2$ where $i, j \in E$
    - $L=BWB^T$ -> B is incidence matrix which appears above!
    - Normalized Laplacian: $L^{sym}_{i,j} := -1/\sqrt{deg(v_i)deg(v_j)}$ if vi is adjacent to vj -> for alleviating one, connected with many vertexes, dominating the structure of graph.
- Directed weighted case
    - There are In-Degree(col sum) matrix and Out-Degree(row sum) matriåx
    - Symmetry Laplacian for a directed graph: $A+A^T$ -> Symmetry Adjacency matrix.

*Related*

- https://en.wikipedia.org/wiki/Laplacian_matrix
- https://stats.stackexchange.com/questions/459640/why-eigenvectors-reveal-the-groups-in-spectral-clustering
- https://math.stackexchange.com/questions/2959261/laplacian-of-a-directed-weighted-graph
- Similarity Graph Clustering: https://www.youtube.com/watch?v=3k9hwRCcT30
- Neural Manifold: https://www.youtube.com/watch?v=QHj9uVmwA_0

*Insights*

- new way to interpret the attention map (in a transition graph view!)
- 

In [ ]:
import networkx as nx

directed_graph = nx.DiGraph()
directed_graph.add_edges_from([(0, 1), (1, 2), (2, 0)])

In [ ]:
awesome_graph(directed_graph, w=np.array([0, 0, 0]))

In [ ]:
an = lambda n: sum([0.9*10**(-i) for i in range(n)])

In [ ]:
an(18)

In [ ]:
an(17)

In [ ]:
an(16)

In [ ]:
an(16)+1

In [ ]:
1e20 + 1